5. Rozdelenie datasetu na trénovaciu a testovaciu množinu

Táto časť pipeline zabezpečuje konzistentné párovanie obrázkov a masiek podľa názvu súboru a následné náhodné rozdelenie na trénovaciu (80 %) a testovaciu (20 %) množinu. Súbory sa kopírujú do samostatných priečinkov pre ďalšie použitie pri trénovaní segmentačného modelu.

In [ ]:
import os
import shutil
import random

# Vstupné priečinky s obrázkami a maskami
INPUT_DIR = 'LOG_STRETCH'
MASK_DIR  = 'LOG_STRETCH_MASKS'

# Výstupné priečinky pre train/test split
TRAIN_INPUT_DIR = 'LOG_STRETCH_TRAIN'
TRAIN_MASK_DIR  = 'LOG_STRETCH_TRAIN_MASK'
TEST_INPUT_DIR  = 'LOG_STRETCH_TEST'
TEST_MASK_DIR   = 'LOG_STRETCH_TEST_MASK'

# Vytvorenie priečinkov, ak neexistujú
for d in [TRAIN_INPUT_DIR, TRAIN_MASK_DIR, TEST_INPUT_DIR, TEST_MASK_DIR]:
    os.makedirs(d, exist_ok=True)

# Zoznam obrázkov a masiek
input_files = sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.png','.jpg','.jpeg'))])
mask_files  = sorted([f for f in os.listdir(MASK_DIR)  if f.lower().endswith(('.png','.jpg','.jpeg'))])

# Dictionary pre bezpečné párovanie podľa názvu
mask_dict = {os.path.basename(f): f for f in mask_files}

# Výber iba tých súborov, ktoré majú masku
paired_files = [f for f in input_files if f in mask_dict]

# Reprodukovateľné premiešanie
random.seed(1234)
random.shuffle(paired_files)

# 80/20 split
split_idx = int(0.8 * len(paired_files))
train_files = paired_files[:split_idx]
test_files  = paired_files[split_idx:]

# Funkcia na kopírovanie párov obrázok–maska
def copy_files(file_list, src_dir, mask_dir, dst_input_dir, dst_mask_dir):
    for f in file_list:
        shutil.copy(os.path.join(src_dir, f), os.path.join(dst_input_dir, f))
        shutil.copy(os.path.join(mask_dir, f), os.path.join(dst_mask_dir, f))

# Kopírovanie train/test množín
copy_files(train_files, INPUT_DIR, MASK_DIR, TRAIN_INPUT_DIR, TRAIN_MASK_DIR)
copy_files(test_files, INPUT_DIR, MASK_DIR, TEST_INPUT_DIR, TEST_MASK_DIR)

print(f"Počet trénovacích obrázkov: {len(train_files)}")
print(f"Počet testovacích obrázkov: {len(test_files)}")
print("Rozdelenie dokončené!")

6. Tréning segmentačného modelu DeepLabV3+

Táto sekcia obsahuje kompletnú implementáciu trénovania segmentačného modelu DeepLabV3+ s ResNet‑50 backbone. Skript zahŕňa:

načítanie a prípravu dát (vrátane augmentácií),

definíciu architektúry,

metriky (Dice, IoU),

checkpointing a ukladanie najlepšieho modelu,

priebežné monitorovanie tréningu,

uloženie finálneho modelu a grafov.

In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Progbar
import matplotlib.pyplot as plt

# -----------------------------------------------------
# ZÁKLADNÉ PARAMETRE TRÉNINGU
# -----------------------------------------------------
IMG_HEIGHT = 256
IMG_WIDTH  = 256
BATCH_SIZE = 8
BUFFER_SIZE = 400
EPOCHS = 30
VAL_SPLIT = 0.2
IMG_EXTS = ('.png', '.jpg', '.jpeg')

# Priečinky s trénovacími dátami
TRAIN_INPUT_DIR = 'CUTOUT_LOG_STRETCH_TRAIN'
TRAIN_MASK_DIR  = 'CUTOUT_LOG_STRETCH_TRAIN_MASK'

# Priečinky pre checkpointy a monitorovanie
MONITOR_DIR = 'epoch_monitor_DeepLabV3plus30_regularization_plots'
CKPT_DIR = "ckpts_DeepLabV3plus30_regularization_plots"

os.makedirs(MONITOR_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

HISTORY_FILE = os.path.join(CKPT_DIR, "training_history.npy")

# -----------------------------------------------------
# FUNKCIE NA NAČÍTANIE OBRÁZKOV A MASIEK
# -----------------------------------------------------
def load_image_and_mask(image_path, mask_path):
    """Načíta obrázok a masku, zmení veľkosť a normalizuje."""
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=1, expand_animations=False)
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = (tf.cast(image, tf.float32) / 127.5) - 1.0  # normalizácia do [-1,1]

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_image(mask, channels=1, expand_animations=False)
    mask = tf.image.resize(mask, [IMG_HEIGHT, IMG_WIDTH], method='nearest')
    mask = tf.cast(mask, tf.float32) / 255.0  # binárna maska

    return image, mask

# -----------------------------------------------------
# AUGMENTÁCIE (flip + rotácia)
# -----------------------------------------------------
def augment_image_and_mask(image, mask):
    """Jednoduché augmentácie pre zvýšenie robustnosti modelu."""
    if tf.random.uniform([]) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)

    if tf.random.uniform([]) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    # Náhodná rotácia o 0, 90, 180 alebo 270 stupňov
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    mask = tf.image.rot90(mask, k)

    return image, mask

# -----------------------------------------------------
# DATASET PIPELINE
# -----------------------------------------------------
def make_dataset_from_paths(input_paths, mask_paths, augment=False):
    """Vytvorí tf.data pipeline pre obrázky a masky."""
    ds = tf.data.Dataset.from_tensor_slices((input_paths, mask_paths))

    ds = ds.map(lambda x, y: load_image_and_mask(x, y),
                num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        ds = ds.map(lambda img, msk: augment_image_and_mask(img, msk),
                    num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# -----------------------------------------------------
# NAČÍTANIE CIEST K OBRÁZKOM A MASKÁM
# -----------------------------------------------------
train_inputs = sorted([os.path.join(TRAIN_INPUT_DIR, f)
                       for f in os.listdir(TRAIN_INPUT_DIR)
                       if f.lower().endswith(IMG_EXTS)])

train_masks = sorted([os.path.join(TRAIN_MASK_DIR, f)
                      for f in os.listdir(TRAIN_MASK_DIR)
                      if f.lower().endswith(IMG_EXTS)])

# Zarovnanie počtu obrázkov a masiek
elem = min(len(train_inputs), len(train_masks))
train_inputs = train_inputs[:elem]
train_masks  = train_masks[:elem]

# Rozdelenie na train/val
split_idx = int((1 - VAL_SPLIT) * elem)

train_ds = make_dataset_from_paths(train_inputs[:split_idx],
                                   train_masks[:split_idx],
                                   augment=True)

val_ds = make_dataset_from_paths(train_inputs[split_idx:],
                                 train_masks[split_idx:],
                                 augment=False)

# -----------------------------------------------------
# DEFINÍCIA ASPP BLOKU (Atrous Spatial Pyramid Pooling)
# -----------------------------------------------------
def ASPP(x, filters=256, l2_reg=1e-4):
    """ASPP modul používaný v DeepLabV3+."""
    y1 = layers.Conv2D(filters,1,padding="same",use_bias=False,
                       kernel_regularizer=regularizers.l2(l2_reg))(x)
    y1 = layers.BatchNormalization()(y1)
    y1 = layers.ReLU()(y1)

    y2 = layers.Conv2D(filters,3,dilation_rate=6,padding="same",
                       use_bias=False,kernel_regularizer=regularizers.l2(l2_reg))(x)
    y2 = layers.BatchNormalization()(y2)
    y2 = layers.ReLU()(y2)

    y3 = layers.Conv2D(filters,3,dilation_rate=12,padding="same",
                       use_bias=False,kernel_regularizer=regularizers.l2(l2_reg))(x)
    y3 = layers.BatchNormalization()(y3)
    y3 = layers.ReLU()(y3)

    y4 = layers.Conv2D(filters,3,dilation_rate=18,padding="same",
                       use_bias=False,kernel_regularizer=regularizers.l2(l2_reg))(x)
    y4 = layers.BatchNormalization()(y4)
    y4 = layers.ReLU()(y4)

    # Globálny priemer pre kontext
    y5 = layers.GlobalAveragePooling2D()(x)
    y5 = layers.Reshape((1,1,-1))(y5)
    y5 = layers.Conv2D(filters,1,padding="same",
                       kernel_regularizer=regularizers.l2(l2_reg))(y5)
    y5 = layers.BatchNormalization()(y5)
    y5 = layers.ReLU()(y5)
    y5 = layers.UpSampling2D(size=(16,16), interpolation='bilinear')(y5)

    # Spojenie vetiev
    y = layers.Concatenate()([y1,y2,y3,y4,y5])

    y = layers.Conv2D(filters,1,padding="same",
                      kernel_regularizer=regularizers.l2(l2_reg))(y)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)

    return y

# -----------------------------------------------------
# DEFINÍCIA MODELU DEEPLABV3+
# -----------------------------------------------------
inputs = layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1))

# ResNet50 očakáva 3 kanály → duplikujeme grayscale
x3 = layers.Concatenate()([inputs,inputs,inputs])

# Backbone ResNet50
base = tf.keras.applications.ResNet50(
    include_top=False,
    weights='imagenet',
    input_tensor=x3
)

# Nízkoúrovňové a vysokoúrovňové feature mapy
low_level = base.get_layer('conv2_block3_out').output
high_level = base.get_layer('conv4_block6_out').output

# ASPP modul
x = ASPP(high_level)

# Upsampling
x = layers.UpSampling2D(size=(4,4), interpolation='bilinear')(x)

# Spracovanie low-level features
low = layers.Conv2D(48,1,padding='same',
                    kernel_regularizer=regularizers.l2(1e-4))(low_level)
low = layers.BatchNormalization()(low)
low = layers.ReLU()(low)

# Spojenie
x = layers.Concatenate()([x,low])

# Dekóder
x = layers.Dropout(0.3)(x)

x = layers.Conv2D(256,3,padding='same',
                  kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)

x = layers.Conv2D(256,3,padding='same',
                  kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)

# Finálny upsampling
x = layers.UpSampling2D(size=(4,4), interpolation='bilinear')(x)

# Výstupná vrstva (logity)
outputs = layers.Conv2D(1,1,activation=None)(x)

model = Model(inputs, outputs)

# -----------------------------------------------------
# DEFINÍCIA LOSS FUNKCIE A METRÍK
# -----------------------------------------------------
def tversky_loss(y_true, y_pred):
    """Tversky loss vhodný pre nevyvážené segmentačné úlohy."""
    y_pred = tf.sigmoid(y_pred)
    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum((1-y_true) * y_pred)
    fn = tf.reduce_sum(y_true * (1-y_pred))
    return 1 - (tp) / (tp + 0.7*fp + 0.3*fn + 1e-6)

def dice_metric(y_true, y_pred):
    """Dice koeficient."""
    y_pred = tf.sigmoid(y_pred)
    inter = tf.reduce_sum(y_true * y_pred)
    return (2*inter) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + 1e-6)

def iou_metric(y_true, y_pred):
    """Intersection over Union."""
    y_pred = tf.sigmoid(y_pred)
    inter = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - inter
    return inter / (union + 1e-6)

optimizer = Adam(1e-4)

# Metriky pre monitorovanie
train_loss_metric = tf.keras.metrics.Mean()
val_loss_metric   = tf.keras.metrics.Mean()
train_dice = tf.keras.metrics.Mean()
val_dice   = tf.keras.metrics.Mean()
train_iou = tf.keras.metrics.Mean()
val_iou   = tf.keras.metrics.Mean()

# -----------------------------------------------------
# CHECKPOINTING
# -----------------------------------------------------
ckpt = tf.train.Checkpoint(model=model,
                           optimizer=optimizer,
                           epoch=tf.Variable(0))

manager = tf.train.CheckpointManager(ckpt, CKPT_DIR, max_to_keep=EPOCHS)

START_EPOCH = 0

# Obnovenie checkpointu, ak existuje
if manager.latest_checkpoint:
    ckpt.restore(manager.latest_checkpoint).expect_partial()
    START_EPOCH = int(ckpt.epoch.numpy())

# História tréningu
if os.path.exists(HISTORY_FILE):
    history = np.load(HISTORY_FILE, allow_pickle=True).item()
else:
    history = {'train_loss':[], 'val_loss':[],
               'train_dice':[], 'val_dice':[],
               'train_iou':[], 'val_iou':[]}

BEST_VAL_DICE = 0.0
BEST_MODEL_PATH = "best_DeepLabV3plus_model.keras"

# -----------------------------------------------------
# TRÉNINGOVÉ KROKY
# -----------------------------------------------------
@tf.function
def train_step(x,y):
    """Jeden tréningový krok."""
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss = tversky_loss(y, logits)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    dice = dice_metric(y, logits)
    iou = iou_metric(y, logits)

    return loss, dice, iou

@tf.function
def val_step(x,y):
    """Jeden validačný krok."""
    logits = model(x, training=False)
    loss = tversky_loss(y, logits)
    dice = dice_metric(y, logits)
    iou = iou_metric(y, logits)
    return loss, dice, iou

# -----------------------------------------------------
# HLAVNÁ TRÉNINGOVÁ SLUČKA
# -----------------------------------------------------
for epoch in range(START_EPOCH, EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    start = time.time()

    # Reset metrík
    train_loss_metric.reset_state()
    val_loss_metric.reset_state()
    train_dice.reset_state()
    val_dice.reset_state()
    train_iou.reset_state()
    val_iou.reset_state()

    # Tréning
    prog_train = Progbar(target=tf.data.experimental.cardinality(train_ds).numpy())
    for step,(x,y) in enumerate(train_ds):
        loss, dice, iou = train_step(x,y)
        train_loss_metric.update_state(loss)
        train_dice.update_state(dice)
        train_iou.update_state(iou)
        prog_train.update(step+1,[('loss',train_loss_metric.result()),
                                  ('dice',train_dice.result()),
                                  ('iou',train_iou.result())])

    # Validácia
    prog_val = Progbar(target=tf.data.experimental.cardinality(val_ds).numpy())
    for step,(x,y) in enumerate(val_ds):
        vloss, vdice, viou = val_step(x,y)
        val_loss_metric.update_state(vloss)
        val_dice.update_state(vdice)
        val_iou.update_state(viou)
        prog_val.update(step+1,[('val_loss',val_loss_metric.result()),
                                ('val_dice',val_dice.result()),
                                ('val_iou',val_iou.result())])

    # Uloženie histórie
    history['train_loss'].append(float(train_loss_metric.result()))
    history['val_loss'].append(float(val_loss_metric.result()))
    history['train_dice'].append(float(train_dice.result()))
    history['val_dice'].append(float(val_dice.result()))
    history['train_iou'].append(float(train_iou.result()))
    history['val_iou'].append(float(val_iou.result()))

    # Uloženie najlepšieho modelu podľa validačného Dice
    current_val_dice = float(val_dice.result())
    if current_val_dice > BEST_VAL_DICE:
        BEST_VAL_DICE = current_val_dice
        model.save(BEST_MODEL_PATH)
        print("New best model saved with val_dice:", BEST_VAL_DICE)

    # Checkpoint
    ckpt.epoch.assign(epoch+1)
    manager.save()

    # Uloženie histórie
    np.save(HISTORY_FILE, history)

    print("Epoch duration:", round(time.time()-start,1),"s")

# -----------------------------------------------------
# VIZUALIZÁCIA PRIEBEHU TRÉNINGU
# -----------------------------------------------------
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(history['train_loss'], label='train_loss')
plt.plot(history['val_loss'], label='val_loss')
plt.title("Loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history['train_dice'], label='train_dice')
plt.plot(history['val_dice'], label='val_dice')
plt.title("Dice")
plt.legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=300)
plt.show()

plt.figure()
plt.plot(history['train_iou'], label='train_iou')
plt.plot(history['val_iou'], label='val_iou')
plt.title("IoU")
plt.legend()
plt.savefig("iou_curve.png", dpi=300)
plt.show()

# Uloženie finálneho modelu
model.save("DeepLabV3plus_final_model_reg.keras")
print("Final model saved")

7. Predikcia masiek na testovacej množine

Táto časť načíta najlepší natrénovaný model, aplikuje ho na testovaciu množinu, pre každú predikovanú masku vykoná základné morfologické čistenie a uloží výsledné binárne masky. Zároveň počíta priemerné hodnoty Dice a IoU voči ground‑truth maskám.

In [ ]:
import os
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import load_model
from scipy.ndimage import binary_fill_holes, label
from skimage.morphology import remove_small_objects, closing, opening, disk
from PIL import Image
from tqdm import tqdm

IMG_HEIGHT = 256
IMG_WIDTH  = 256
IMG_EXTS   = ('.png', '.jpg', '.jpeg')

TEST_INPUT_DIR = 'CUTOUT_LOG_STRETCH_TEST'
TEST_MASK_DIR  = 'CUTOUT_LOG_STRETCH_TEST_MASK'
OUTPUT_DIR     = "PREDICTED_MASKS"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Načítavam model...")
seg_model = load_model('best_DeepLabV3plus_model.keras', compile=False)
print("Model načítaný.")

def load_image(image_path):
    """Načíta a normalizuje vstupný obrázok."""
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=1, expand_animations=False)
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = (tf.cast(image, tf.float32) / 127.5) - 1.0
    return image

def load_mask(mask_path):
    """Načíta a normalizuje ground-truth masku."""
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_image(mask, channels=1, expand_animations=False)
    mask = tf.image.resize(mask, [IMG_HEIGHT, IMG_WIDTH], method='nearest')
    mask = tf.cast(mask, tf.float32) / 255.0
    return mask

def clean_mask(mask, min_size=300, smooth_radius=2):
    """Základné morfologické čistenie predikovanej masky."""
    mask = mask.astype(bool)
    mask = binary_fill_holes(mask)
    mask = remove_small_objects(mask, min_size=min_size)

    labeled, num = label(mask)
    if num > 0:
        sizes = np.bincount(labeled.ravel())
        sizes[0] = 0
        largest = np.argmax(sizes)
        mask = (labeled == largest)

    selem = disk(smooth_radius)
    mask = closing(mask, selem)
    mask = opening(mask, selem)

    return mask.astype(np.uint8)

def dice_score(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    return (2 * inter) / (np.sum(y_true) + np.sum(y_pred) + 1e-6)

def iou_score(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - inter
    return inter / (union + 1e-6)

# Zoznam testovacích obrázkov a masiek
test_inputs = [
    os.path.join(TEST_INPUT_DIR, f)
    for f in os.listdir(TEST_INPUT_DIR)
    if f.lower().endswith(IMG_EXTS)
]

test_masks = [
    os.path.join(TEST_MASK_DIR, f)
    for f in os.listdir(TEST_MASK_DIR)
    if f.lower().endswith(IMG_EXTS)
]

mask_dict = {os.path.basename(p): p for p in test_masks}
test_inputs = [p for p in test_inputs if os.path.basename(p) in mask_dict]

print(f"Počet obrázkov na predikciu: {len(test_inputs)}")

threshold = 0.5
dice_scores = []
iou_scores  = []

for img_path in tqdm(test_inputs, desc="Predicting", unit="image"):

    filename = os.path.basename(img_path)
    mask_path = mask_dict[filename]

    img = load_image(img_path)
    img = tf.expand_dims(img, axis=0)

    # Predikcia a prahovanie
    pred = seg_model(img, training=False)
    pred_mask = tf.sigmoid(pred)[0, ..., 0].numpy()
    binary_mask = (pred_mask > threshold).astype(np.float32)

    # Morfologické čistenie
    binary_mask = clean_mask(binary_mask)

    # Ground-truth maska
    gt_mask = load_mask(mask_path).numpy()[..., 0]
    gt_mask = (gt_mask > 0.5).astype(np.float32)

    # Metriky
    dice = dice_score(gt_mask, binary_mask)
    iou  = iou_score(gt_mask, binary_mask)

    dice_scores.append(dice)
    iou_scores.append(iou)

    # Uloženie predikovanej masky
    save_path = os.path.join(OUTPUT_DIR, filename)
    mask_img = (binary_mask * 255).astype(np.uint8)
    Image.fromarray(mask_img).save(save_path)

mean_dice = np.mean(dice_scores)
mean_iou  = np.mean(iou_scores)

print("\nHotovo.")
print(f"Masky uložené v: {OUTPUT_DIR}")
print(f"Priemerný Dice: {mean_dice:.4f}")
print(f"Priemerný IoU : {mean_iou:.4f}")

8. Zjednotenie trénovacích a predikovaných masiek

Táto časť spája všetky dostupné masky (ručne/automaticky vytvorené pre tréning a predikované pre test) do jedného priečinka. Párovanie prebieha podľa názvu galaxie odvodeného z FITS súborov.

In [ ]:
import os
import shutil

TRAIN_MASK_DIR = "CUTOUT_LOG_STRETCH_TRAIN_MASK"
PRED_MASK_DIR  = "PREDICTED_MASKS"
FITS_DIR       = "FITS_VIS_LOC_EXTRACTED"
OUTPUT_DIR     = "MASKS_ALL"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def base_name_first_dot(filename):
    """Vráti časť názvu pred prvou bodkou."""
    return filename.split(".")[0]

# Vytvorenie mapy názov → cesta k maske
mask_dict = {}
for folder in [TRAIN_MASK_DIR, PRED_MASK_DIR]:
    for f in os.listdir(folder):
        key = base_name_first_dot(f)
        mask_dict[key] = os.path.join(folder, f)

# Zoznam FITS súborov, podľa ktorých sa páruje
fits_files = sorted([f for f in os.listdir(FITS_DIR) if f.endswith(".fits")])

copied = 0
missing = []

for fits_file in fits_files:
    galaxy_name = base_name_first_dot(fits_file)

    if galaxy_name in mask_dict:
        src = mask_dict[galaxy_name]
        dst = os.path.join(OUTPUT_DIR, os.path.basename(src))
        shutil.copy(src, dst)
        copied += 1
    else:
        missing.append(galaxy_name)

print("Hotovo")
print("Počet skopírovaných masiek:", copied)

if missing:
    print("Chýbajú masky pre tieto galaxie:")
    for m in missing:
        print(m)